In [3]:
import pandas as pd
import yfinance as yf
import time
import os

PROCESSED_DATA_DIR = "../../data/processed"

In [4]:
# Cell 1 (updated to load the clean file)
equity_prices = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "equity_close_prices_clean.csv"), index_col=0, parse_dates=True)
mapping = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "cusip_ticker_sector_mapping_final.csv"), dtype={"cusip": str})

print(equity_prices.shape)
print(mapping.shape)
mapping.head()

(755, 3451)
(41065, 4)


,cusip,ticker,security_type,matched_name
0,000000008,NaN,NaN,NaN
1,000000018,NaN,NaN,NaN
2,000000nan,NaN,NaN,NaN
3,000003128,NaN,NaN,NaN
4,000003638,NaN,NaN,NaN


In [5]:
equity_prices = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "equity_close_prices_clean.csv"), index_col=0, parse_dates=True)
print(equity_prices.shape)

tickers = equity_prices.columns.tolist()
print(f"Tickers to source sector data for: {len(tickers)}")

(755, 3451)
Tickers to source sector data for: 3451


In [6]:
def fetch_sector_info(ticker_list, checkpoint_path=None, checkpoint_every=100, max_retries=2):
    results = {}
    total = len(ticker_list)

    for i, ticker in enumerate(ticker_list):
        sector, industry = None, None
        for attempt in range(max_retries):
            try:
                info = yf.Ticker(ticker).info
                sector = info.get("sector") or None
                industry = info.get("industry") or None
                break
            except Exception:
                time.sleep(2)

        results[ticker] = {"sector": sector, "industry": industry}

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1} / {total}")

        if checkpoint_path and (i + 1) % checkpoint_every == 0:
            pd.DataFrame.from_dict(results, orient="index").reset_index().rename(
                columns={"index": "ticker"}).to_csv(checkpoint_path, index=False)

    return results

In [7]:
sector_results = fetch_sector_info(
    tickers,
    checkpoint_path=os.path.join(PROCESSED_DATA_DIR, "sector_mapping_checkpoint.csv")
)

sector_df = pd.DataFrame.from_dict(sector_results, orient="index").reset_index().rename(columns={"index": "ticker"})
sector_df.to_csv(os.path.join(PROCESSED_DATA_DIR, "sector_mapping.csv"), index=False)

missing = sector_df["sector"].isna().sum()
print(f"\nTickers with no sector data: {missing} / {len(sector_df)} ({missing/len(sector_df):.1%})")

Processed 100 / 3451
Processed 200 / 3451
Processed 300 / 3451
Processed 400 / 3451
Processed 500 / 3451
Processed 600 / 3451
Processed 700 / 3451
Processed 800 / 3451
Processed 900 / 3451
Processed 1000 / 3451
Processed 1100 / 3451
Processed 1200 / 3451
Processed 1300 / 3451
Processed 1400 / 3451
Processed 1500 / 3451
Processed 1600 / 3451
Processed 1700 / 3451
Processed 1800 / 3451
Processed 1900 / 3451
Processed 2000 / 3451
Processed 2100 / 3451
Processed 2200 / 3451
Processed 2300 / 3451
Processed 2400 / 3451
Processed 2500 / 3451
Processed 2600 / 3451
Processed 2700 / 3451
Processed 2800 / 3451
Processed 2900 / 3451
Processed 3000 / 3451
Processed 3100 / 3451
Processed 3200 / 3451
Processed 3300 / 3451
Processed 3400 / 3451

Tickers with no sector data: 33 / 3451 (1.0%)


In [8]:
# Quick sanity check
print(sector_df["sector"].value_counts())
print(f"\nTotal unique sectors: {sector_df['sector'].nunique()}")

sector
Financial Services        654
Industrials               510
Consumer Cyclical         388
Technology                379
Healthcare                376
Basic Materials           269
Consumer Defensive        212
Energy                    177
Real Estate               174
Communication Services    171
Utilities                 108
Name: count, dtype: int64

Total unique sectors: 11


In [9]:
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "cleaned_holdings.csv"), dtype={"cusip": str})
print(df.shape)

(1630738, 15)


In [10]:
mapping = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "cusip_ticker_sector_mapping_final.csv"), dtype={"cusip": str})
print(mapping.columns.tolist())

['cusip', 'ticker', 'security_type', 'matched_name']


In [11]:
df_merged = df.merge(mapping[["cusip", "ticker", "security_type"]], on="cusip", how="left")

equity_security_types = ["Common Stock", "Depositary Receipt", "REIT", "Preferred Stock"]
df_equity = df_merged[df_merged["security_type"].isin(equity_security_types)].copy()

print(f"Holdings after merging with mapping: {len(df_merged)}")
print(f"Holdings restricted to equity security types: {len(df_equity)}")

df_equity = df_equity.merge(sector_df[["ticker", "sector", "industry"]], on="ticker", how="left")

usable_tickers = set(equity_prices.columns)
df_equity = df_equity[df_equity["ticker"].isin(usable_tickers)].copy()

print(f"Holdings restricted to tickers with usable price data: {len(df_equity)}")
print(f"\nMissing sector after merge: {df_equity['sector'].isna().sum()} / {len(df_equity)}")

df_equity.head()

Holdings after merging with mapping: 1630738
Holdings restricted to equity security types: 1009933
Holdings restricted to tickers with usable price data: 741422

Missing sector after merge: 6311 / 741422


,cusip,cik,2015-03,2015-06,2015-09,2015-12,2016-03,2016-06,2016-09,2016-12,2017-03,2017-06,2017-09,stock,institution,ticker,security_type,sector,industry
1,000304105,714142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,AAC TECHNOLOGIES HLDGS INC,TEACHERS RETIREMENT SYSTEM OF THE STATE OF KEN...,AACAY,Depositary Receipt,Technology,Communication Equipment
2,000304105,820027,0.0,0.0,0.0,0.0,0.0,3084.0,2662.0,2492.0,2260.0,2260.0,2260.0,AAC TECHNOLOGIES H-UNSPON AD,AMERICAN EXPRESS FINANCIAL ADVISORS,AACAY,Depositary Receipt,Technology,Communication Equipment
3,000304105,883782,0.0,0.0,0.0,0.0,0.0,85.0,71.0,4641.0,21.0,21.0,21.0,AAC TECHNOLOGIES HLDGS INC,FULTON BANK,AACAY,Depositary Receipt,Technology,Communication Equipment
4,000304105,932859,0.0,13207.0,14285.0,14285.0,17385.0,28309.0,26919.0,26539.0,26093.0,29879.0,31817.0,AAC ACOUSTIC TECHNOLOG ADR,PARAMETRIC PORTFOLIO ASSOCIATES,AACAY,Depositary Receipt,Technology,Communication Equipment
5,000304105,944234,0.0,0.0,0.0,0.0,0.0,189282.0,188244.0,335136.0,152263.0,152263.0,152263.0,AAC TECHNOLOGIES HOLDINGS IN,RENAISSANCE INVESTMENT MANAGEMENT,AACAY,Depositary Receipt,Technology,Communication Equipment


In [12]:
# Load everything needed, fresh, using the corrected (forward-fill-fixed) data
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "cleaned_holdings.csv"), dtype={"cusip": str})
mapping = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "cusip_ticker_sector_mapping_final.csv"), dtype={"cusip": str})
sector_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "sector_mapping.csv"))

print(f"Holdings: {df.shape}")
print(f"Mapping: {mapping.shape}")
print(f"Sector: {sector_df.shape}")

Holdings: (1630738, 15)
Mapping: (41065, 4)
Sector: (3451, 3)


In [13]:
# Merge holdings -> mapping -> sector, filter to usable equity holdings
df_merged = df.merge(mapping[["cusip", "ticker", "security_type"]], on="cusip", how="left")

equity_security_types = ["Common Stock", "Depositary Receipt", "REIT", "Preferred Stock"]
df_equity = df_merged[df_merged["security_type"].isin(equity_security_types)].copy()

df_equity = df_equity.merge(sector_df[["ticker", "sector", "industry"]], on="ticker", how="left")

usable_tickers = set(equity_prices.columns)
df_equity = df_equity[df_equity["ticker"].isin(usable_tickers)].copy()

print(f"After mapping merge: {len(df_merged)}")
print(f"After equity-type filter: {len(df_equity)}")
print(f"After usable-ticker filter: {len(df_equity)}")
print(f"Missing sector: {df_equity['sector'].isna().sum()} / {len(df_equity)}")

After mapping merge: 1630738
After equity-type filter: 741422
After usable-ticker filter: 741422
Missing sector: 6311 / 741422


In [14]:
quarter_cols = ["2015-03", "2015-06", "2015-09", "2015-12",
                 "2016-03", "2016-06", "2016-09", "2016-12",
                 "2017-03", "2017-06", "2017-09"]

# Melt wide (one column per quarter) into long format (one row per institution-holding-quarter)
df_long = df_equity.melt(
    id_vars=["cusip", "cik", "institution", "stock", "ticker", "security_type", "sector", "industry"],
    value_vars=quarter_cols,
    var_name="quarter",
    value_name="shares"
)

# Drop rows where the institution didn't actually hold anything that quarter
df_long = df_long[df_long["shares"] > 0].copy()
print(f"Long-format holding-quarter rows (non-zero shares): {len(df_long)}")

# Map each quarter label to the actual quarter-end date, then find the closest available trading price on or before it
quarter_end_dates = {
    "2015-03": "2015-03-31", "2015-06": "2015-06-30", "2015-09": "2015-09-30", "2015-12": "2015-12-31",
    "2016-03": "2016-03-31", "2016-06": "2016-06-30", "2016-09": "2016-09-30", "2016-12": "2016-12-31",
    "2017-03": "2017-03-31", "2017-06": "2017-06-30", "2017-09": "2017-09-30"
}
df_long["quarter_end_date"] = pd.to_datetime(df_long["quarter"].map(quarter_end_dates))

# Build a price lookup: for each ticker, the closest available price on or before each quarter-end
price_lookup = {}
for q_label, q_date in quarter_end_dates.items():
    q_date = pd.Timestamp(q_date)
    available_dates = equity_prices.index[equity_prices.index <= q_date]
    if len(available_dates) > 0:
        closest_date = available_dates.max()
        price_lookup[q_label] = equity_prices.loc[closest_date]
    else:
        price_lookup[q_label] = pd.Series(dtype=float)

df_long["price"] = df_long.apply(lambda row: price_lookup[row["quarter"]].get(row["ticker"], None), axis=1)
df_long["dollar_value"] = df_long["shares"] * df_long["price"]

print(f"Rows with valid price: {df_long['price'].notna().sum()} / {len(df_long)}")
df_long.head()

Long-format holding-quarter rows (non-zero shares): 5375433
Rows with valid price: 5362478 / 5375433


,cusip,cik,institution,stock,ticker,security_type,sector,industry,quarter,shares,quarter_end_date,price,dollar_value
173,000360206,1558481,Arizona State Retirement System,AAON INC,AAON,Common Stock,Industrials,Building Products & Equipment,2015-03,29275.0,2015-03-31,15.176843,4.443021e+05
245,000361105,867262,GW CAPITAL INC,AAR CORP,AIR,Common Stock,Industrials,Aerospace & Defense,2015-03,729591.0,2015-03-31,29.228424,2.132480e+07
375,000361105,1511566,Spot Trading L.L.C,AAR CORP,AIR,Common Stock,Industrials,Aerospace & Defense,2015-03,17100.0,2015-03-31,29.228424,4.998061e+05
389,000361105,1558481,Arizona State Retirement System,AAR CORP,AIR,Common Stock,Industrials,Aerospace & Defense,2015-03,26523.0,2015-03-31,29.228424,7.752255e+05
530,000375204,1037763,GRAVER BOKHOF GOODWILL & SULLIVAN L P/IL,ABB LTD,ABBNY,Depositary Receipt,Industrials,Electrical Equipment & Parts,2015-03,1450.0,2015-03-31,14.078145,2.041331e+04


In [15]:
# Portfolio-level features per institution per quarter, computed only on rows with a valid price
df_valid = df_long[df_long["price"].notna()].copy()

def compute_portfolio_features(group):
    total_value = group["dollar_value"].sum()
    weights = group["dollar_value"] / total_value

    hhi = (weights ** 2).sum()
    top_holding_pct = weights.max()
    num_holdings = len(group)

    sector_values = group.groupby("sector")["dollar_value"].sum()
    sector_weights = sector_values / total_value
    sector_hhi = (sector_weights ** 2).sum()

    return pd.Series({
        "total_value": total_value,
        "hhi": hhi,
        "top_holding_pct": top_holding_pct,
        "num_holdings": num_holdings,
        "sector_hhi": sector_hhi
    })

portfolio_features = df_valid.groupby(["cik", "institution", "quarter"]).apply(
    compute_portfolio_features, include_groups=False
).reset_index()

print(f"Institution-quarter portfolios: {len(portfolio_features)}")
portfolio_features.head()

Institution-quarter portfolios: 22855


,cik,institution,quarter,total_value,hhi,top_holding_pct,num_holdings,sector_hhi
0,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-06,7.765856e+08,0.022841,0.050151,63.0,0.138630
1,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-09,7.264922e+08,0.022167,0.040915,65.0,0.133333
2,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-12,8.025855e+08,0.022105,0.050331,69.0,0.130072
3,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-03,8.241263e+08,0.020321,0.049144,72.0,0.125416
4,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-06,8.775283e+08,0.019110,0.043060,75.0,0.128084


In [16]:
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
portfolio_features.head()

,cik,institution,quarter,total_value,hhi,top_holding_pct,num_holdings,sector_hhi
0,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-06,"776,585,645.59",0.02,0.05,63.00,0.14
1,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-09,"726,492,197.65",0.02,0.04,65.00,0.13
2,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-12,"802,585,459.78",0.02,0.05,69.00,0.13
3,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-03,"824,126,284.19",0.02,0.05,72.00,0.13
4,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-06,"877,528,261.70",0.02,0.04,75.00,0.13


In [17]:
portfolio_features.to_csv(os.path.join(PROCESSED_DATA_DIR, "portfolio_features.csv"), index=False)
print(f"Portfolio features saved: {portfolio_features.shape}")

Portfolio features saved: (22855, 8)


In [20]:
benchmark = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "sp500_benchmark.csv"), header=[0, 1], index_col=0)
benchmark.index = pd.to_datetime(benchmark.index)

print(benchmark.columns.tolist())
benchmark.head()

[('Close', '^GSPC'), ('High', '^GSPC'), ('Low', '^GSPC'), ('Open', '^GSPC'), ('Volume', '^GSPC')]


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2015-01-02,"2,058.20","2,072.36","2,046.04","2,058.90",2708700000
2015-01-05,"2,020.58","2,054.44","2,017.34","2,054.44",3799120000
2015-01-06,"2,002.61","2,030.25","1,992.44","2,022.15",4460110000
2015-01-07,"2,025.90","2,029.61","2,005.55","2,005.55",3805480000
2015-01-08,"2,062.14","2,064.08","2,030.61","2,030.61",3934010000


In [21]:
benchmark_close = benchmark[("Close", "^GSPC")]
benchmark_close.index = pd.to_datetime(benchmark_close.index)

print(benchmark_close.head())

Date
2015-01-02   2,058.20
2015-01-05   2,020.58
2015-01-06   2,002.61
2015-01-07   2,025.90
2015-01-08   2,062.14
Name: (Close, ^GSPC), dtype: float64


In [22]:
quarter_order = ["2015-03", "2015-06", "2015-09", "2015-12",
                  "2016-03", "2016-06", "2016-09", "2016-12",
                  "2017-03", "2017-06", "2017-09"]
next_quarter_map = {quarter_order[i]: quarter_order[i+1] for i in range(len(quarter_order)-1)}

def get_price_on_or_before(price_series, date):
    available = price_series.index[price_series.index <= date]
    return price_series.loc[available.max()] if len(available) > 0 else None

def compute_forward_return(ticker, current_q_date, next_q_date):
    try:
        price_now = get_price_on_or_before(equity_prices[ticker], current_q_date)
        price_next = get_price_on_or_before(equity_prices[ticker], next_q_date)
        if price_now is None or price_next is None or price_now == 0:
            return None
        return (price_next - price_now) / price_now
    except KeyError:
        return None

def benchmark_return(current_q_date, next_q_date):
    price_now = get_price_on_or_before(benchmark_close, current_q_date)
    price_next = get_price_on_or_before(benchmark_close, next_q_date)
    return (price_next - price_now) / price_now

label_rows = []

for quarter in quarter_order[:-1]:  # exclude the last quarter, no "next quarter" to measure
    next_quarter = next_quarter_map[quarter]
    current_q_date = pd.Timestamp(quarter_end_dates[quarter])
    next_q_date = pd.Timestamp(quarter_end_dates[next_quarter])

    bench_ret = benchmark_return(current_q_date, next_q_date)

    quarter_holdings = df_valid[df_valid["quarter"] == quarter]

    for (cik, institution), group in quarter_holdings.groupby(["cik", "institution"]):
        total_value = group["dollar_value"].sum()
        weights = group["dollar_value"] / total_value

        weighted_excess_return = 0
        for _, row in group.iterrows():
            fwd_ret = compute_forward_return(row["ticker"], current_q_date, next_q_date)
            if fwd_ret is not None:
                weight = row["dollar_value"] / total_value
                weighted_excess_return += weight * (fwd_ret - bench_ret)

        label_rows.append({
            "cik": cik, "institution": institution, "quarter": quarter,
            "weighted_excess_return": weighted_excess_return
        })

label_df = pd.DataFrame(label_rows)
print(f"Label rows computed: {len(label_df)}")
label_df.head()

Label rows computed: 20244


,cik,institution,quarter,weighted_excess_return
0,50672,INSTITUTIONAL CAPITAL CORP,2015-03,0.01
1,867262,GW CAPITAL INC,2015-03,-0.01
2,939835,AMERICAN NATIONAL BANK & TRUST CO /VA,2015-03,NaN
3,949623,FINANCIAL COUNSELORS INC,2015-03,NaN
4,1033244,NYL TRUST CO,2015-03,0.00


In [23]:
print(f"Total NaN rows: {label_df['weighted_excess_return'].isna().sum()} / {len(label_df)}")

# Investigate one specific NaN case
nan_example = label_df[label_df["weighted_excess_return"].isna()].iloc[0]
print(f"\nInvestigating: CIK {nan_example['cik']}, quarter {nan_example['quarter']}")

example_group = df_valid[(df_valid["cik"] == nan_example["cik"]) & (df_valid["quarter"] == nan_example["quarter"])]
print(f"Number of holdings in this institution-quarter: {len(example_group)}")
print(example_group[["ticker", "dollar_value"]])

Total NaN rows: 1610 / 20244

Investigating: CIK 939835, quarter 2015-03
Number of holdings in this institution-quarter: 83
       ticker  dollar_value
5275        T    375,483.82
8037      ABT    857,145.82
9806     ABBV  1,206,338.15
21447     APD    294,857.47
36860      MO    313,156.11
...       ...           ...
695057     VZ  1,836,207.68
703303    VOD    605,304.62
709362    WMT     62,784.58
715808    WFC    350,980.81
724776    WHR    197,670.89

[83 rows x 2 columns]


In [24]:
print(f"total_value for this group: {example_group['dollar_value'].sum()}")
print(f"Any NaN in dollar_value column for this group: {example_group['dollar_value'].isna().sum()}")
print(f"Dtype of dollar_value: {example_group['dollar_value'].dtype}")

# Check if any of these 83 tickers have duplicate rows within this same group,
# which combined with iterrows() could behave unexpectedly
print(f"\nDuplicate tickers within this institution-quarter: {example_group['ticker'].duplicated().sum()}")

total_value for this group: 78582906.88547087
Any NaN in dollar_value column for this group: 0
Dtype of dollar_value: float64

Duplicate tickers within this institution-quarter: 0


In [25]:
for ticker in example_group["ticker"].unique():
    fwd = compute_forward_return(ticker, pd.Timestamp("2015-03-31"), pd.Timestamp("2015-06-30"))
    if fwd is not None and pd.isna(fwd):
        print(f"{ticker}: returns NaN (not None) - this is the bug")

TEV: returns NaN (not None) - this is the bug


In [26]:
def compute_forward_return(ticker, current_q_date, next_q_date):
    try:
        price_now = get_price_on_or_before(equity_prices[ticker], current_q_date)
        price_next = get_price_on_or_before(equity_prices[ticker], next_q_date)
        if price_now is None or price_next is None:
            return None
        if pd.isna(price_now) or pd.isna(price_next) or price_now == 0:
            return None
        return (price_next - price_now) / price_now
    except KeyError:
        return None

In [27]:
def benchmark_return(current_q_date, next_q_date):
    price_now = get_price_on_or_before(benchmark_close, current_q_date)
    price_next = get_price_on_or_before(benchmark_close, next_q_date)
    if price_now is None or price_next is None or pd.isna(price_now) or pd.isna(price_next):
        return None
    return (price_next - price_now) / price_now

In [28]:
label_rows = []

for quarter in quarter_order[:-1]:
    next_quarter = next_quarter_map[quarter]
    current_q_date = pd.Timestamp(quarter_end_dates[quarter])
    next_q_date = pd.Timestamp(quarter_end_dates[next_quarter])

    bench_ret = benchmark_return(current_q_date, next_q_date)
    if bench_ret is None:
        print(f"Skipping {quarter}: no valid benchmark return")
        continue

    quarter_holdings = df_valid[df_valid["quarter"] == quarter]

    for (cik, institution), group in quarter_holdings.groupby(["cik", "institution"]):
        total_value = group["dollar_value"].sum()

        weighted_excess_return = 0
        any_valid = False
        for _, row in group.iterrows():
            fwd_ret = compute_forward_return(row["ticker"], current_q_date, next_q_date)
            if fwd_ret is not None:
                weight = row["dollar_value"] / total_value
                weighted_excess_return += weight * (fwd_ret - bench_ret)
                any_valid = True

        label_rows.append({
            "cik": cik, "institution": institution, "quarter": quarter,
            "weighted_excess_return": weighted_excess_return if any_valid else None
        })

label_df = pd.DataFrame(label_rows)
print(f"Label rows computed: {len(label_df)}")
print(f"NaN/None rows: {label_df['weighted_excess_return'].isna().sum()} / {len(label_df)}")

Label rows computed: 20244
NaN/None rows: 0 / 20244


In [29]:
loss_threshold = -0.15  # 15% or worse underperformance relative to benchmark

label_df["label"] = (label_df["weighted_excess_return"] < loss_threshold).astype(int)

print(label_df["label"].value_counts())
print(f"\nPositive class (concentration-amplified loss) proportion: {label_df['label'].mean():.1%}")

print("\nDistribution of weighted_excess_return:")
print(label_df["weighted_excess_return"].describe())

label
0    20110
1      134
Name: count, dtype: int64

Positive class (concentration-amplified loss) proportion: 0.7%

Distribution of weighted_excess_return:
count   20,244.00
mean         0.01
std          0.05
min         -1.04
25%         -0.01
50%          0.01
75%          0.02
max          0.79
Name: weighted_excess_return, dtype: float64


In [30]:
for threshold in [-0.05, -0.07, -0.10, -0.12]:
    proportion = (label_df["weighted_excess_return"] < threshold).mean()
    count = (label_df["weighted_excess_return"] < threshold).sum()
    print(f"Threshold {threshold:.0%}: {count} positive cases ({proportion:.1%})")

Threshold -5%: 854 positive cases (4.2%)
Threshold -7%: 516 positive cases (2.5%)
Threshold -10%: 283 positive cases (1.4%)
Threshold -12%: 189 positive cases (0.9%)


In [31]:
loss_threshold = -0.05
label_df["label"] = (label_df["weighted_excess_return"] < loss_threshold).astype(int)

print(label_df["label"].value_counts())
print(f"Positive class proportion: {label_df['label'].mean():.1%}")

# Merge portfolio features with the label
model_data = portfolio_features.merge(
    label_df[["cik", "quarter", "weighted_excess_return", "label"]],
    on=["cik", "quarter"],
    how="inner"
)

print(f"\nFinal modeling table: {model_data.shape}")
print(model_data["label"].value_counts())
model_data.head()

label
0    19390
1      854
Name: count, dtype: int64
Positive class proportion: 4.2%

Final modeling table: (20244, 10)
label
0    19390
1      854
Name: count, dtype: int64


,cik,institution,quarter,total_value,hhi,top_holding_pct,num_holdings,sector_hhi,weighted_excess_return,label
0,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-06,"776,585,645.59",0.02,0.05,63.00,0.14,-0.01,0
1,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-09,"726,492,197.65",0.02,0.04,65.00,0.13,-0.01,0
2,2230,ADAMS DIVERSIFIED EQUITY FUND,2015-12,"802,585,459.78",0.02,0.05,69.00,0.13,-0.01,0
3,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-03,"824,126,284.19",0.02,0.05,72.00,0.13,-0.00,0
4,2230,ADAMS DIVERSIFIED EQUITY FUND,2016-06,"877,528,261.70",0.02,0.04,75.00,0.13,0.00,0


In [32]:
model_data.to_csv(os.path.join(PROCESSED_DATA_DIR, "model_features.csv"), index=False)
print("Final feature + label table saved.")

Final feature + label table saved.
